In [1]:
import pandas as pd
from pathlib import Path
import plotly.express as px
import plotly.io as pio
import numpy as np
import plotly.graph_objects as go

cdhits =['99','95','90','85','80','75','70','65','60']
pathogen='dysgalactiae'
BASE_DIR = Path(
    "../../../data/manuscript/dysgalactiae"
)

In [2]:
unique = []
accessory = []
core = []
nbr_matches_genes = []


tot_genomes=193

limit_percent=0.02
lower_limit=tot_genomes * limit_percent
upper_limit=tot_genomes * (1-limit_percent)

for cdhit in cdhits:

    cdhit_path = (
        BASE_DIR
        / f"cdhit_{cdhit}"
        / f"{pathogen}_cdhit_{cdhit}_cluster_frame.csv"
    )

    df = pd.read_csv(
            cdhit_path,
            index_col=0
        )

    nbr_genomes = df["nbr_genomes"]

    unique.append(
        (nbr_genomes < lower_limit).sum()
    )
    accessory.append(
        (
            (nbr_genomes >= lower_limit)
            & (nbr_genomes <= upper_limit)
        ).sum()
    )

    core.append(
        (nbr_genomes > upper_limit).sum()
    )

    nbr_matches_genes.append(
        len(df)
    )

# ============================================================
# Create summary dataframe
# ============================================================

genome = pd.DataFrame({
    "cdhit": cdhits,
    "Unique": unique,
    "Accessory": accessory,
    "Core": core,
    "Total": nbr_matches_genes
})

genome = genome.set_index("cdhit")


# ============================================================
# Select and reorder columns
# ============================================================

genome = genome[
    [
        "Core",
        "Accessory",
        "Unique"
    ]
]

df = pd.concat(
    [genome],
    axis=1,
    keys=["Genome"]
)
# ============================================================
# Coverage labels
# ============================================================

coverage = {
    "Core": ">98%",
    "Accessory": "2-98%",
    "Unique": "<2%"
}


In [3]:
max_genome = max(nbr_matches_genes)

colors = {
    "Genome": {
        "Core": "#054a91",
        "Accessory": "#81a4cd",
        "Unique": "#dbe4ee",
    }
}
# Create a figure with the right layout for genome data only
fig_genome_narrow = go.Figure(
    layout=go.Layout(
        height=800,
        width=650,
        barmode="relative",
        yaxis=dict(
            showticklabels=True,
            showgrid=True,
            range=[0, max_genome + 500],
            title=dict(
                text="Gene sequences",
                font=dict(size=18)
            ),
            tickfont=dict(size=14)
        ),
        font=dict(size=16),
        legend=dict(
            title=dict(
                text="Genome",
                font=dict(size=19)
            ),
            x=0.5,
            y=-0.25,
            xanchor="center",
            yanchor="top",
            orientation="v",
            font=dict(size=16)
        ),
        hovermode="x",
        margin=dict(b=40, t=40, l=40, r=40)
    )
)

# Add the traces for genome data only with narrower bars
legend_names = {
    "Core": "Core genome (>98%)",
    "Accessory": "Accessory genome (2-98%)",
    "Unique": "Unique genome (<2%)"
}

for j, col in enumerate(df["Genome"].columns):
    fig_genome_narrow.add_bar(
        x=df.index,
        y=df["Genome"][col],
        name=legend_names[col],
        marker=dict(
            color=colors["Genome"][col],
            line=dict(
                color="#8a8a8a",
                width=1
            )
        ),
        width=0.65,
        hovertemplate="%{y}<extra></extra>"
    )

# Update layout to set white background and x-axis title
fig_genome_narrow.update_layout(
    plot_bgcolor='white',
    paper_bgcolor='white',
    xaxis=dict(
        title=dict(
            text="CD-HIT sequence identity threshold (%)",
            font=dict(size=18),
            standoff=35
        ),
        tickfont=dict(
            size=30,
            family='Arial, bold'

        )
    ),
    yaxis=dict(
        showticklabels=True,
        showgrid=True,
        range=[0, max_genome + 500],
        tickformat=",d",
        title=dict(
            text="Gene sequences",
            font=dict(size=18)
        ),
        tickfont=dict(size=20)
    )
)

fig_genome_narrow.show()